# New Methods Showcase: Sessions 85-97

This tutorial demonstrates the **7 new method families** added in Sessions 85-97:

1. **Selection** - Heckman two-stage for sample selection bias
2. **Bounds** - Manski bounds when assumptions fail
3. **QTE** - Quantile treatment effects (distributional)
4. **MTE** - Marginal treatment effects across propensity
5. **Mediation** - Direct vs indirect effects
6. **Control Function** - Alternative to IV for endogeneity
7. **Shift-Share** - Bartik instruments with diagnostics

Each section demonstrates when to use the method, its key assumptions, and interpretation.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys
sys.path.insert(0, '../../src')

# Import new method families
from causal_inference.selection import heckman_two_stage
from causal_inference.bounds import manski_bounds, lee_bounds
from causal_inference.qte import unconditional_qte, conditional_qte
from causal_inference.mte import marginal_treatment_effect, policy_relevant_te
from causal_inference.mediation import mediation_analysis, sensitivity_analysis
from causal_inference.control_function import control_function
from causal_inference.shift_share import shift_share_iv, rotemberg_weights

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

---

# Part 1: Selection (Heckman Two-Stage)

**When to use**: Sample selection is non-random and correlated with the outcome.

**Example**: Wages are only observed for employed people. If ability affects both employment and wages, OLS on observed wages is biased.

**Key output**: Lambda (λ) - the Mills ratio coefficient indicates selection bias severity.

In [ ]:
def generate_selection_data(n=2000, selection_strength=0.5, seed=42):
    """
    Generate data with sample selection.
    
    Scenario: Wage regression with employment selection.
    - Ability affects both employment (selection) and wages (outcome)
    - Education is the variable of interest
    - Marriage is exclusion restriction (affects employment, not wages)
    """
    np.random.seed(seed)
    
    # Covariates
    ability = np.random.randn(n)  # Unobserved
    education = 12 + 2 * np.random.randn(n)
    married = np.random.binomial(1, 0.5, n)  # Exclusion restriction
    
    # Selection equation (employment)
    # Married people more likely to work (exclusion restriction)
    z_select = -0.5 + 0.1 * education + selection_strength * ability + 0.8 * married
    prob_employed = 1 / (1 + np.exp(-z_select))
    employed = np.random.binomial(1, prob_employed)
    
    # Outcome equation (wages) - only for employed
    wage = 10 + 0.5 * education + 0.8 * ability + np.random.randn(n)
    wage_observed = np.where(employed == 1, wage, np.nan)
    
    return pd.DataFrame({
        'wage': wage_observed,
        'education': education,
        'married': married,
        'employed': employed,
        'true_wage': wage,
        'ability': ability
    })

df_select = generate_selection_data(n=2000, selection_strength=0.5)

print("Sample Selection Data:")
print(f"  Total observations: {len(df_select)}")
print(f"  Employed (observed wages): {df_select['employed'].sum()}")
print(f"  Not employed (missing wages): {(~df_select['employed'].astype(bool)).sum()}")
print(f"  Selection rate: {df_select['employed'].mean():.1%}")

In [ ]:
# Compare OLS (biased) vs Heckman (corrected)
print("=== Selection Bias Demonstration ===")
print(f"True effect of education on wages: 0.5")
print()

# OLS on observed wages only (biased)
observed = df_select[df_select['employed'] == 1]
X_ols = observed[['education']].values
X_ols = np.column_stack([np.ones(len(X_ols)), X_ols])
y_ols = observed['wage'].values
beta_ols = np.linalg.lstsq(X_ols, y_ols, rcond=None)[0]
print(f"OLS estimate (biased): {beta_ols[1]:.4f}")

# Heckman two-stage
result_heckman = heckman_two_stage(
    Y=df_select['wage'].values,
    X=df_select[['education']].values,
    Z=df_select[['education', 'married']].values,  # Exclusion: married
    selection=df_select['employed'].values
)

print(f"Heckman estimate: {result_heckman['beta'][1]:.4f}")
print(f"")
print(f"Selection correction (lambda): {result_heckman['lambda']:.4f}")
print(f"Lambda p-value: {result_heckman['lambda_pvalue']:.4f}")
print(f"")
print(f"Interpretation:")
if result_heckman['lambda_pvalue'] < 0.05:
    print(f"  Lambda is significant → Selection bias present")
    print(f"  Lambda > 0 → Positive selection (high ability → employed AND high wages)")
else:
    print(f"  Lambda not significant → No evidence of selection bias")

### Selection Key Takeaways

1. **Lambda (λ)** measures selection bias severity
2. **Exclusion restriction** required: variable affecting selection but not outcome
3. **Compare OLS vs Heckman** - if estimates differ substantially, selection matters

---

# Part 2: Bounds (Manski & Lee)

**When to use**: Standard identification assumptions fail, but you want honest inference.

**Key insight**: Bounds trade off assumptions for width. Fewer assumptions = wider bounds.

**Types**:
- **Manski worst-case**: No assumptions, widest bounds
- **Manski monotone**: Assumes treatment response is monotone
- **Lee bounds**: For sample selection (attrition)

In [ ]:
def generate_bounds_data(n=1000, true_effect=2.0, seed=42):
    """
    Generate data where standard assumptions may fail.
    Bounded outcome (0-10 scale) for Manski bounds.
    """
    np.random.seed(seed)
    
    # Treatment assignment with some selection
    X = np.random.randn(n)
    prob_treat = 1 / (1 + np.exp(-0.5 * X))
    T = np.random.binomial(1, prob_treat)
    
    # Potential outcomes
    Y0 = 5 + 0.5 * X + np.random.randn(n)
    Y1 = Y0 + true_effect
    
    # Observed outcome
    Y = np.where(T == 1, Y1, Y0)
    Y = np.clip(Y, 0, 10)  # Bounded outcome
    
    return Y, T, X

Y_bounds, T_bounds, X_bounds = generate_bounds_data(n=1000, true_effect=2.0)
print(f"Bounds data: {len(Y_bounds)} observations")
print(f"Treatment rate: {T_bounds.mean():.1%}")
print(f"Outcome range: [{Y_bounds.min():.2f}, {Y_bounds.max():.2f}]")

In [ ]:
# Manski bounds under different assumptions
print("=== Manski Bounds ===")
print(f"True effect: 2.0")
print(f"Outcome support: [0, 10]")
print()

# Worst-case (no assumptions)
result_worst = manski_bounds(
    Y=Y_bounds, T=T_bounds, 
    y_lower=0, y_upper=10,
    assumption='none'
)
print(f"Worst-case bounds (no assumptions): [{result_worst['lower']:.3f}, {result_worst['upper']:.3f}]")
print(f"  Width: {result_worst['upper'] - result_worst['lower']:.3f}")

# Monotone treatment response (MTR)
result_mtr = manski_bounds(
    Y=Y_bounds, T=T_bounds,
    y_lower=0, y_upper=10,
    assumption='mtr'  # Treatment weakly increases outcome
)
print(f"MTR bounds (treatment helps): [{result_mtr['lower']:.3f}, {result_mtr['upper']:.3f}]")
print(f"  Width: {result_mtr['upper'] - result_mtr['lower']:.3f}")

# Monotone treatment selection (MTS)
result_mts = manski_bounds(
    Y=Y_bounds, T=T_bounds,
    y_lower=0, y_upper=10,
    assumption='mts'  # Those who select treatment have higher Y(1)
)
print(f"MTS bounds (selection on gains): [{result_mts['lower']:.3f}, {result_mts['upper']:.3f}]")
print(f"  Width: {result_mts['upper'] - result_mts['lower']:.3f}")

In [ ]:
# Visualize bounds
fig, ax = plt.subplots(figsize=(10, 5))

bounds_data = [
    ('Worst-case', result_worst['lower'], result_worst['upper']),
    ('MTR', result_mtr['lower'], result_mtr['upper']),
    ('MTS', result_mts['lower'], result_mts['upper']),
]

y_pos = np.arange(len(bounds_data))
for i, (name, lower, upper) in enumerate(bounds_data):
    ax.barh(i, upper - lower, left=lower, height=0.5, alpha=0.7, label=name)
    ax.plot([lower, upper], [i, i], 'k-', linewidth=2)
    ax.plot(lower, i, 'ko', markersize=8)
    ax.plot(upper, i, 'ko', markersize=8)

ax.axvline(x=2.0, color='red', linestyle='--', label='True effect')
ax.set_yticks(y_pos)
ax.set_yticklabels([b[0] for b in bounds_data])
ax.set_xlabel('Treatment Effect')
ax.set_title('Manski Bounds: Assumptions Tighten Bounds')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Bounds Key Takeaways

1. **Honest inference** - bounds never give false precision
2. **More assumptions → tighter bounds** but risk invalidity
3. **Check if zero is excluded** - bounds excluding 0 give significance

---

# Part 3: QTE (Quantile Treatment Effects)

**When to use**: Care about distributional effects, not just the mean.

**Key insight**: Treatment might help the median but hurt the extremes.

**Types**:
- **Unconditional QTE**: Effect on population quantiles
- **Conditional QTE**: Effect at quantile given covariates

In [ ]:
def generate_qte_data(n=1000, seed=42):
    """
    Generate data with heterogeneous effects across distribution.
    
    Treatment compresses the distribution:
    - Helps low earners more
    - Helps high earners less
    """
    np.random.seed(seed)
    
    T = np.random.binomial(1, 0.5, n)
    X = np.random.randn(n)
    
    # Base outcome
    epsilon = np.random.randn(n)
    Y0 = 50 + 10 * X + 15 * epsilon  # High variance
    
    # Treatment effect varies across distribution
    # Compresses: large positive effect for low Y, small for high Y
    tau = 10 - 5 * (epsilon / epsilon.std())  # Effect decreases with rank
    Y1 = Y0 + tau
    
    Y = np.where(T == 1, Y1, Y0)
    
    return Y, T, X, tau

Y_qte, T_qte, X_qte, true_tau = generate_qte_data(n=2000)

print(f"QTE data: {len(Y_qte)} observations")
print(f"ATE: {true_tau.mean():.2f}")
print(f"Effect at 10th percentile: ~{np.percentile(true_tau, 90):.2f} (helps low earners)")
print(f"Effect at 90th percentile: ~{np.percentile(true_tau, 10):.2f} (helps high earners less)")

In [ ]:
# Estimate QTE across quantiles
quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]

result_qte = unconditional_qte(
    Y=Y_qte, T=T_qte,
    quantiles=quantiles
)

print("=== Quantile Treatment Effects ===")
print(f"ATE (mean): {np.mean(Y_qte[T_qte==1]) - np.mean(Y_qte[T_qte==0]):.3f}")
print()
print(f"{'Quantile':>10} {'QTE':>10} {'SE':>10} {'95% CI':>20}")
print("-" * 55)
for q, est, se in zip(result_qte['quantiles'], result_qte['estimates'], result_qte['ses']):
    ci_low = est - 1.96 * se
    ci_high = est + 1.96 * se
    print(f"{q:>10.0%} {est:>10.3f} {se:>10.3f} [{ci_low:>7.3f}, {ci_high:>7.3f}]")

In [ ]:
# Visualize QTE across quantiles
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(result_qte['quantiles'], result_qte['estimates'],
            yerr=1.96 * np.array(result_qte['ses']),
            fmt='o-', capsize=4, markersize=8, label='QTE')

# Add ATE reference line
ate = np.mean(Y_qte[T_qte==1]) - np.mean(Y_qte[T_qte==0])
ax.axhline(y=ate, color='red', linestyle='--', label=f'ATE = {ate:.2f}')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

ax.set_xlabel('Quantile')
ax.set_ylabel('Treatment Effect')
ax.set_title('QTE: Distributional Treatment Effects\n(Treatment compresses the distribution)')
ax.legend()
ax.set_xticks(quantiles)
ax.set_xticklabels([f'{q:.0%}' for q in quantiles])
plt.tight_layout()
plt.show()

### QTE Key Takeaways

1. **ATE can miss important heterogeneity** across the distribution
2. **QTE(τ) ≠ Effect on τ-quantile group** - it's the effect on population quantiles
3. **Policy implications**: A program might help inequality even if ATE is small

---

# Part 4: MTE (Marginal Treatment Effects)

**When to use**: IV setting, want to understand heterogeneity across propensity to treat.

**Key insight**: MTE(p) = effect for those indifferent at propensity p.
- LATE is a weighted average of MTEs
- Allows evaluation of new policies

In [ ]:
def generate_mte_data(n=2000, seed=42):
    """
    Generate data with heterogeneous effects across propensity.
    
    Those most likely to take treatment (high propensity) 
    have larger treatment effects (selection on gains).
    """
    np.random.seed(seed)
    
    # Instrument
    Z = np.random.randn(n)
    
    # Covariates
    X = np.random.randn(n)
    
    # Unobserved heterogeneity in treatment gains
    U = np.random.uniform(0, 1, n)  # Uniformized unobserved
    
    # Propensity score
    propensity = 0.5 + 0.3 * Z + 0.1 * X
    propensity = np.clip(propensity, 0.05, 0.95)
    
    # Treatment selection: take treatment if U < propensity
    T = (U < propensity).astype(int)
    
    # MTE decreases with U (selection on gains)
    # Those most willing to treat (low U) have highest effects
    mte_true = 5 - 3 * U  # MTE ranges from 5 (U=0) to 2 (U=1)
    
    # Outcomes
    Y0 = 10 + 0.5 * X + np.random.randn(n)
    Y1 = Y0 + mte_true
    Y = np.where(T == 1, Y1, Y0)
    
    return Y, T, Z, X, mte_true, propensity

Y_mte, T_mte, Z_mte, X_mte, true_mte, propensity = generate_mte_data(n=2000)

print(f"MTE data: {len(Y_mte)} observations")
print(f"Treatment rate: {T_mte.mean():.1%}")
print(f"True MTE at p=0: {5:.2f}")
print(f"True MTE at p=1: {2:.2f}")
print(f"ATE: {true_mte.mean():.2f}")

In [ ]:
# Estimate MTE curve
result_mte = marginal_treatment_effect(
    Y=Y_mte, T=T_mte, Z=Z_mte, X=X_mte,
    propensity_grid=np.linspace(0.1, 0.9, 9)
)

print("=== Marginal Treatment Effects ===")
print(f"{'Propensity':>12} {'MTE':>10} {'SE':>10}")
print("-" * 35)
for p, mte, se in zip(result_mte['propensity'], result_mte['mte'], result_mte['se']):
    print(f"{p:>12.2f} {mte:>10.3f} {se:>10.3f}")

In [ ]:
# Visualize MTE curve
fig, ax = plt.subplots(figsize=(10, 6))

# Estimated MTE
ax.plot(result_mte['propensity'], result_mte['mte'], 'bo-', 
        markersize=8, label='Estimated MTE')
ax.fill_between(result_mte['propensity'],
                np.array(result_mte['mte']) - 1.96 * np.array(result_mte['se']),
                np.array(result_mte['mte']) + 1.96 * np.array(result_mte['se']),
                alpha=0.3)

# True MTE
p_grid = np.linspace(0, 1, 100)
true_mte_curve = 5 - 3 * p_grid
ax.plot(p_grid, true_mte_curve, 'r--', label='True MTE')

ax.set_xlabel('Propensity Score')
ax.set_ylabel('Marginal Treatment Effect')
ax.set_title('MTE: Effects Vary by Propensity\n(Selection on gains: eager participants benefit more)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

### MTE Key Takeaways

1. **MTE(p) shows who benefits** at each propensity level
2. **Declining MTE** = selection on gains (most eager benefit most)
3. **Policy evaluation**: Can predict effects of policies that change propensity

---

# Part 5: Mediation Analysis

**When to use**: Want to decompose total effect into direct and indirect paths.

**Framework**:
- **Total Effect (TE)** = Direct Effect (NDE) + Indirect Effect (NIE)
- **NDE**: Effect if mediator held at control value
- **NIE**: Effect through the mediator pathway

**Example**: Job training → Skills → Wages

In [ ]:
def generate_mediation_data(n=1000, seed=42):
    """
    Generate data with mediation structure.
    
    T (training) → M (skills) → Y (wages)
           ↘_____________↗ (direct effect)
    """
    np.random.seed(seed)
    
    # Treatment (job training)
    T = np.random.binomial(1, 0.5, n)
    
    # Covariate (prior ability)
    X = np.random.randn(n)
    
    # Mediator (skills) affected by treatment
    # T → M coefficient = 2.0
    M = 5 + 2.0 * T + 0.5 * X + np.random.randn(n)
    
    # Outcome (wages) affected by treatment and mediator
    # Direct effect: T → Y = 1.0
    # Indirect (through M): T → M → Y = 2.0 * 0.8 = 1.6
    Y = 20 + 1.0 * T + 0.8 * M + 0.3 * X + np.random.randn(n)
    
    return Y, T, M, X

Y_med, T_med, M_med, X_med = generate_mediation_data(n=2000)

print("Mediation data:")
print(f"  True direct effect (NDE): 1.0")
print(f"  True T→M: 2.0")
print(f"  True M→Y: 0.8")
print(f"  True indirect effect (NIE): 2.0 × 0.8 = 1.6")
print(f"  True total effect: 1.0 + 1.6 = 2.6")

In [ ]:
# Run mediation analysis
result_med = mediation_analysis(
    Y=Y_med, T=T_med, M=M_med, X=X_med.reshape(-1, 1)
)

print("=== Mediation Analysis Results ===")
print(f"")
print(f"{'Effect':>20} {'Estimate':>10} {'SE':>10} {'p-value':>10}")
print("-" * 55)
print(f"{'Total Effect':>20} {result_med['total_effect']:.4f} {result_med['te_se']:.4f} {result_med['te_pvalue']:.4f}")
print(f"{'Direct (NDE)':>20} {result_med['nde']:.4f} {result_med['nde_se']:.4f} {result_med['nde_pvalue']:.4f}")
print(f"{'Indirect (NIE)':>20} {result_med['nie']:.4f} {result_med['nie_se']:.4f} {result_med['nie_pvalue']:.4f}")
print(f"")
print(f"Proportion mediated: {result_med['proportion_mediated']:.1%}")
print(f"")
print(f"Interpretation:")
print(f"  {result_med['proportion_mediated']:.0%} of the effect operates through skills (mediator)")
print(f"  {1-result_med['proportion_mediated']:.0%} is a direct effect of training")

In [ ]:
# Visualize effect decomposition
fig, ax = plt.subplots(figsize=(8, 6))

effects = ['Total Effect', 'Direct (NDE)', 'Indirect (NIE)']
values = [result_med['total_effect'], result_med['nde'], result_med['nie']]
errors = [result_med['te_se'], result_med['nde_se'], result_med['nie_se']]
true_values = [2.6, 1.0, 1.6]
colors = ['steelblue', 'coral', 'mediumseagreen']

x = np.arange(len(effects))
width = 0.35

bars1 = ax.bar(x - width/2, values, width, yerr=[1.96*e for e in errors], 
               capsize=5, label='Estimated', color=colors, alpha=0.8)
bars2 = ax.bar(x + width/2, true_values, width, 
               label='True', color=colors, alpha=0.4, edgecolor='black', linewidth=2)

ax.set_ylabel('Effect Size')
ax.set_title('Mediation: Direct vs Indirect Effects')
ax.set_xticks(x)
ax.set_xticklabels(effects)
ax.legend()
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity analysis for unmeasured confounding
print("=== Sensitivity Analysis ===")
sens_result = sensitivity_analysis(
    Y=Y_med, T=T_med, M=M_med, X=X_med.reshape(-1, 1)
)

print(f"Sensitivity parameter (rho): correlation between mediator and outcome residuals")
print(f"")
print(f"NIE estimates at different rho values:")
for rho, nie in zip(sens_result['rho_values'], sens_result['nie_at_rho']):
    print(f"  rho = {rho:+.2f}: NIE = {nie:.3f}")
print(f"")
print(f"NIE = 0 at rho = {sens_result['rho_at_zero']:.3f}")
print(f"Interpretation: Would need correlation of {sens_result['rho_at_zero']:.2f} with")
print(f"                unmeasured confounders to nullify the indirect effect.")

### Mediation Key Takeaways

1. **Total = Direct + Indirect** - decomposition reveals mechanism
2. **Proportion mediated** - how much operates through the mediator
3. **Sensitivity analysis** - how robust to unmeasured confounding
4. **Sequential ignorability** - the critical assumption (T→M and M→Y both unconfounded)

---

# Part 6: Control Function

**When to use**: Endogeneity problem with known functional form.

**Key insight**: Include first-stage residual as control for endogeneity.

**Advantages over 2SLS**:
- Works with nonlinear models (probit, logit)
- Can handle heterogeneous treatment effects
- Directly tests for endogeneity (significance of residual)

In [ ]:
def generate_cf_data(n=1000, seed=42):
    """
    Generate data with endogeneity.
    
    Education is endogenous (correlated with ability).
    Distance to college is instrument.
    """
    np.random.seed(seed)
    
    # Instrument (distance to college)
    Z = np.random.randn(n)
    
    # Unobserved (ability)
    U = np.random.randn(n)
    
    # Covariates
    X = np.random.randn(n)
    
    # Endogenous treatment (education)
    # Affected by instrument and unobserved ability
    education = 12 + 0.5 * Z + 0.3 * X + 0.6 * U + np.random.randn(n) * 0.5
    
    # Outcome (wages)
    # True effect of education = 0.5
    wages = 20 + 0.5 * education + 0.3 * X + 0.8 * U + np.random.randn(n)
    
    return wages, education, Z, X

Y_cf, T_cf, Z_cf, X_cf = generate_cf_data(n=2000)

print("Control Function data:")
print(f"  True effect: 0.5")
print(f"  Endogeneity: Education correlated with ability (omitted)")
print(f"  Instrument: Distance to college")

In [ ]:
# Compare OLS, 2SLS, and Control Function
print("=== Comparing Approaches ===")
print(f"True effect: 0.5")
print()

# OLS (biased)
X_ols = np.column_stack([np.ones(len(Y_cf)), T_cf, X_cf])
beta_ols = np.linalg.lstsq(X_ols, Y_cf, rcond=None)[0]
print(f"OLS estimate (biased): {beta_ols[1]:.4f}")

# 2SLS
# First stage
X_1s = np.column_stack([np.ones(len(Y_cf)), Z_cf, X_cf])
gamma = np.linalg.lstsq(X_1s, T_cf, rcond=None)[0]
T_hat = X_1s @ gamma
# Second stage
X_2s = np.column_stack([np.ones(len(Y_cf)), T_hat, X_cf])
beta_2sls = np.linalg.lstsq(X_2s, Y_cf, rcond=None)[0]
print(f"2SLS estimate: {beta_2sls[1]:.4f}")

# Control Function
result_cf = control_function(
    Y=Y_cf, T=T_cf, Z=Z_cf, X=X_cf.reshape(-1, 1)
)
print(f"Control Function estimate: {result_cf['estimate']:.4f}")
print()
print(f"Control function residual coefficient: {result_cf['cf_coefficient']:.4f}")
print(f"Residual p-value: {result_cf['cf_pvalue']:.4f}")
print()
print(f"Interpretation:")
if result_cf['cf_pvalue'] < 0.05:
    print(f"  Residual significant → Endogeneity present (OLS biased)")
else:
    print(f"  Residual not significant → No evidence of endogeneity")

### Control Function Key Takeaways

1. **Alternative to 2SLS** with more flexibility for nonlinear models
2. **Built-in endogeneity test** - significance of control residual
3. **Same identification** as IV - requires valid exclusion restriction

---

# Part 7: Shift-Share IV (Bartik)

**When to use**: Industry/sector exposure varies across locations + aggregate shocks.

**Instrument construction**: Z_i = Σ_s (share_{i,s} × shock_s)

**Example**: Labor market effects of trade shocks using industry composition.

In [ ]:
def generate_shift_share_data(n_regions=100, n_sectors=10, seed=42):
    """
    Generate shift-share data.
    
    Scenario: Effect of import competition on employment.
    - Regions have different industry compositions (shares)
    - Industries face different import shocks
    - Bartik instrument = sum of (share × shock)
    """
    np.random.seed(seed)
    
    # Industry shares for each region (sum to 1)
    raw_shares = np.random.exponential(1, (n_regions, n_sectors))
    shares = raw_shares / raw_shares.sum(axis=1, keepdims=True)
    
    # National-level import shocks by sector
    shocks = np.random.randn(n_sectors) * 0.5
    
    # Bartik instrument: exposure to shocks
    bartik = shares @ shocks
    
    # Treatment (import competition exposure)
    # Affected by Bartik and local factors
    local = np.random.randn(n_regions)
    T = 0.5 * bartik + 0.3 * local + np.random.randn(n_regions) * 0.2
    
    # Outcome (employment change)
    # True effect of import competition = -2.0
    Y = 5 - 2.0 * T + 0.5 * local + np.random.randn(n_regions)
    
    return Y, T, shares, shocks, bartik

Y_ss, T_ss, shares_ss, shocks_ss, bartik_ss = generate_shift_share_data(
    n_regions=200, n_sectors=15
)

print("Shift-Share data:")
print(f"  Regions: 200")
print(f"  Sectors: 15")
print(f"  True effect: -2.0 (import competition hurts employment)")

In [ ]:
# Run shift-share IV
result_ss = shift_share_iv(
    Y=Y_ss, T=T_ss, 
    shares=shares_ss, 
    shocks=shocks_ss
)

print("=== Shift-Share IV Results ===")
print(f"True effect: -2.0")
print()
print(f"Estimate: {result_ss['estimate']:.4f}")
print(f"Std Error: {result_ss['se']:.4f}")
print(f"95% CI: [{result_ss['ci_lower']:.4f}, {result_ss['ci_upper']:.4f}]")
print(f"")
print(f"First-stage F-statistic: {result_ss['first_stage_f']:.2f}")

In [ ]:
# Rotemberg weights diagnostics
rotemberg = rotemberg_weights(
    Y=Y_ss, T=T_ss,
    shares=shares_ss,
    shocks=shocks_ss
)

print("=== Rotemberg Diagnostics ===")
print(f"")
print(f"Negative weight share: {rotemberg['negative_weight_share']:.1%}")
print(f"  (High % = potential monotonicity violations)")
print(f"")
print(f"Top 5 sectors by weight:")
print(f"{'Sector':>8} {'Weight':>10} {'Shock':>10} {'Contribution':>12}")
print("-" * 45)
for sector, weight, shock, contrib in rotemberg['top_5_sectors']:
    print(f"{sector:>8} {weight:>10.3f} {shock:>10.3f} {contrib:>12.3f}")

print(f"")
print(f"Interpretation:")
print(f"  Top sector contributes {abs(rotemberg['top_5_sectors'][0][3]):.1%} of estimate")
print(f"  If that sector's shock is invalid, estimate may be biased")

In [ ]:
# Visualize Rotemberg weights
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Weight distribution
ax1.bar(range(len(rotemberg['weights'])), sorted(rotemberg['weights'], reverse=True))
ax1.axhline(y=0, color='red', linestyle='--')
ax1.set_xlabel('Sector (sorted by weight)')
ax1.set_ylabel('Rotemberg Weight')
ax1.set_title('Sector Weights in Shift-Share Estimate')

# Shock vs weight scatter
ax2.scatter(shocks_ss, rotemberg['weights'], alpha=0.7)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Sector Shock')
ax2.set_ylabel('Rotemberg Weight')
ax2.set_title('Shock Magnitude vs Sector Weight')

plt.tight_layout()
plt.show()

### Shift-Share Key Takeaways

1. **Two identification strategies**: exogenous shares OR exogenous shocks
2. **Rotemberg weights** reveal which sectors drive the estimate
3. **Negative weights** signal potential monotonicity violations
4. **Sensitivity**: If top sectors are questionable, estimate may be fragile

---

# Summary: When to Use Each Method

| Method | Use When | Key Output |
|--------|----------|------------|
| **Selection (Heckman)** | Sample selection is non-random | Lambda (Mills ratio) |
| **Bounds (Manski/Lee)** | Standard assumptions fail | [lower, upper] bounds |
| **QTE** | Care about distributional effects | QTE(τ) curve |
| **MTE** | IV setting, want heterogeneity by propensity | MTE(p) curve |
| **Mediation** | Decompose direct vs indirect effects | NDE, NIE, % mediated |
| **Control Function** | Endogeneity, nonlinear models | Estimate + endogeneity test |
| **Shift-Share** | Sector exposure varies, aggregate shocks | Estimate + Rotemberg diagnostics |

### Decision Guidance

1. **Missing data not at random?** → Heckman Selection
2. **Worried about assumptions?** → Manski Bounds
3. **Care about inequality?** → QTE
4. **Who benefits most from IV?** → MTE
5. **What's the mechanism?** → Mediation
6. **Endogeneity in nonlinear model?** → Control Function
7. **Industry exposure + shocks?** → Shift-Share IV

---

*Created: Session 98 - Causal Inference Mastery*